# District Population Forecasting for Primary-School Classroom Planning

Five districts (Kampala, Wakiso, Gulu, plus Mbale and Mbarara added for comparison) analysed using 10 years of annual population estimates (2015–2024, thousands).

All reusable logic lives in `src/`; this notebook only imports it and shows results.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make `src` importable whether the notebook is opened from the project root or elsewhere
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.population import load_districts, YEARS
from src.stats import descriptive_stats, yoy_growth, cagr
from src.models import MODEL_CLASSES
from src.validation import validate_models, forecast_districts
from src.planning import classroom_plan, SCHOOL_AGE_SHARE, PUPILS_PER_CLASSROOM
from src.bootstrap import bootstrap_all
from src.plotting import plot_forecasts

rng = np.random.RandomState(42)   # single seeded stream -> reproducible bootstrap
FORECAST_YEARS = np.arange(2025, 2030)

## 1. DistrictPopulation objects

In [ ]:
districts = load_districts()
for d in districts.values():
    print(repr(d))

## 2. Descriptive statistics — `statistics` vs NumPy

In [ ]:
rows = []
for name, d in districts.items():
    s, n = descriptive_stats(d)
    rows.append({
        "District": name, "Mean": s["mean"], "Median": s["median"],
        "Var (statistics, ddof=1)": s["variance"], "Var (np, ddof=0)": n["variance_pop"],
        "Std (statistics)": s["stdev"], "Std (np, ddof=0)": n["std_pop"],
    })
pd.DataFrame(rows).set_index("District").round(1)

**Why they differ.** `statistics.variance()` / `stdev()` always divide by **n − 1** (Bessel's correction: the data is treated as a *sample*). `np.var()` / `np.std()` default to dividing by **n** (`ddof=0`, the data is the *whole population*). `ddof` sets the divisor to `n − ddof`, so `np.var(x, ddof=1)` reproduces `statistics.variance(x)` exactly. With n = 10 the sample variance is 10/9 ≈ 1.11× the population variance; the gap shrinks as n grows.

## 3. Year-on-year growth and CAGR

In [ ]:
cagr_results = {name: cagr(d.populations) for name, d in districts.items()}
for name, d in districts.items():
    print(f"{name:10s} YoY% = {np.round(yoy_growth(d.populations), 2)}")

pd.Series({k: round(v * 100, 2) for k, v in cagr_results.items()}, name="CAGR 2015-2024 (%)").to_frame()

In [ ]:
fastest = max(cagr_results, key=cagr_results.get)
print(f"Fastest-growing in relative terms: {fastest} ({cagr_results[fastest]*100:.2f}% CAGR)")

## 4–5. Forecasting models and train/test validation

Train on 2015–2021, test on 2022–2024. Best model per district = lowest RMSE.

In [ ]:
validation_tables, best_models = validate_models(districts, split_year=2021)

val_df = pd.DataFrame(
    [{"District": n, **r, "Selected": r["model"] == best_models[n]}
     for n, rows in validation_tables.items() for r in rows]
).set_index(["District", "model"])
val_df.round(2)

In [ ]:
best_models

## 6. 2025–2029 forecasts and variance comparison

In [ ]:
forecasts, models = forecast_districts(districts, best_models, FORECAST_YEARS)

fc_df = pd.DataFrame(
    {name: np.round(f, 1) for name, f in forecasts.items()}, index=FORECAST_YEARS
).T
fc_df.insert(0, "Model", pd.Series(best_models))
fc_df["Var(actual)"] = [np.var(districts[n].populations, ddof=1) for n in fc_df.index]
fc_df["Var(forecast)"] = [np.var(forecasts[n], ddof=1) for n in fc_df.index]
fc_df.round(1)

**Var(forecast) < Var(actual) for every district.** The historical series mixes genuine year-to-year irregularity (migration, boundary changes, measurement noise) with the trend; the fitted model is a noise-free smooth curve, and five points of it are mechanically tighter than ten years of real growth. This is an **artefact of model smoothness, not evidence that future population will vary less** — hence the prediction intervals below.

## Extension A — Bootstrap prediction intervals

In [ ]:
intervals = bootstrap_all(districts, models, horizon=len(FORECAST_YEARS), rng=rng, n_boot=1000)
for name, (lo, hi) in intervals.items():
    print(f"{name} ({best_models[name]}) 95% PI at 2029: [{lo[-1]:.0f}, {hi[-1]:.0f}]k")

Bands are narrow because the selected fits track history closely. They are a **lower bound on true uncertainty**: they capture only historical fitting error, not the risk that the underlying growth *rate* changes (boundary reclassification, infrastructure, policy) — the biggest real-world risk for a 5-year schools plan.

## 7. Combined plot

In [ ]:
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

fig = plot_forecasts(districts, best_models, models, forecasts, intervals,
                     FORECAST_YEARS, split_year=2021,
                     save_path=OUTPUT_DIR / "district_forecasts.png")
plt.show()

## 8. Classroom planning (school-age share 18%, 53 pupils per classroom)

In [ ]:
plan = classroom_plan(districts, forecasts)
pd.DataFrame(plan).T.rename(columns={
    "pop_2024": "2024 pop (k)", "pop_2029": "2029 pop (k)",
    "rooms_2024": "Classrooms 2024", "rooms_2029": "Classrooms 2029",
    "additional": "Additional needed",
}).round({"2024 pop (k)": 0, "2029 pop (k)": 0}).astype(
    {"Classrooms 2024": int, "Classrooms 2029": int, "Additional needed": int})

## Extension B — Critique of the Fibonacci-ratio model

The model scales the last value by successive Fibonacci ratios (1, 2, 1.5, 1.667, 1.6, …), which converge to the golden ratio φ ≈ 1.618. As a year-on-year multiplier that implies **~62% growth every year**, compounding — versus real annual growth of 2–7% here. Validation makes this obvious: MAPE of 190–215% on 2022–2024, i.e. predictions two to three times too high within three years.

It is only "defensible" in the coincidental sense that any fixed multiplicative growth model has a similar form; a district compounding at ~62% a year would double every 14–15 months, which does not happen. It has no link to demographic drivers (births, deaths, migration, urbanisation), so it is a cautionary example of fitting an elegant sequence to a problem it has no mechanistic relationship to. Train/test validation is exactly what catches this before it reaches a planning decision.